# Global House Purchase Decisions

In [ ]:
# ===============================================================
# Title: Global House Purchase Decision Prediction - Advanced Analysis
# Group: Zamira Dzhatdoyev
# Objective: Predict purchase decisions with novel features and insights
# ===============================================================

# 0. Import Libraries

In [ ]:
# Import libraries for:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Baseline models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Ensemble / boosting
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# Neural network
from sklearn.neural_network import MLPClassifier

# Metrics
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

# Handling imbalance
from imblearn.over_sampling import SMOTE

# SHAP
import shap

# 1. Load Dataset

(Using github repo to access dataset)



In [ ]:
url = "https://raw.githubusercontent.com/zdzhatdo/global-house-decision-project/refs/heads/main/global_house_purchase_dataset.csv"

# Numeric columns with correct types
dtype_dict = {
    'property_id': 'Int64',
    'property_size_sqft': 'Int64',
    'price': 'Int64',
    'constructed_year': 'Int64',
    'previous_owners': 'Int64',
    'rooms': 'Int64',
    'bathrooms': 'Int64',
    'garage': 'Int64',
    'garden': 'Int64',
    'crime_cases_reported': 'Int64',
    'legal_cases_on_property': 'Int64',
    'customer_salary': 'Int64',
    'loan_amount': 'Int64',
    'loan_tenure_years': 'Int64',
    'monthly_expenses': 'Int64',
    'down_payment': 'Int64',
    'satisfaction_score': 'Int64',
    'neighbourhood_rating': 'Int64',
    'connectivity_score': 'Int64',
    'decision': 'Int64',
    'emi_to_income_ratio': 'float'  # the only float
}

In [ ]:
# Load dataset and preview first few rows
df = pd.read_csv(url, dtype=dtype_dict)
print(df.dtypes)

# Display basic info and statistics
print(df.head())
print(df['constructed_year'].unique()[:10]) # sanity check

# 2. Initial Exploration - Zamira

In [ ]:
# shape, columns
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# quick head
display(df.head())

# dtypes summary
print("\nDtypes:\n", df.dtypes)

# missing values
print("\nMissing values per column:\n", df.isna().sum())

# unique counts for categorical columns
cat_cols = ['country', 'city', 'property_type', 'furnishing_status']
for c in cat_cols:
    if c in df.columns:
        print(f"\nColumn '{c}': unique={df[c].nunique()}, top='{df[c].mode().iloc[0]}' (freq={df[c].value_counts().iloc[0]})")

# target distribution (counts & percent)
print("\nTarget counts:\n", df['decision'].value_counts())
print("\nTarget proportions:\n", df['decision'].value_counts(normalize=True))

# basic sanity checks (sample)
print("\nconstructed_year min/max:", df['constructed_year'].min(), df['constructed_year'].max())
print("price min/max:", df['price'].min(), df['price'].max())
print("customer_salary min/max:", df['customer_salary'].min(), df['customer_salary'].max())


NOTES:
<br>
<br> Shape: there are 200,000 points and 25 attributes.
<br> Numeric summary: observed mean, 75%, max values of categories.
<br> Categorical: observed top country, top city, top property type, and top property status
<br>
<br> Target variable:
<br> 0 = "Not buy" -> 153,932 samples where the model should learn not to buy the property
<br> 1 = "Buy" -> 46,068 samples where the model should learn that the property is worth buying
<br> Proportions:
<br> 0 = Not Buy -> 76.966%
<br> 1 = Buy -> 23.034%
<br>
<br> The dataset is imbalanced. The model could guess "Not Buy" all the time and still get 77% accuracy, which is misleading. This means that accuracy alone is not a good metric of performance.
<br> Need: class balancing, additional metrics of performance
<br> Buying is the minority event: the model must learn a good pattern to predict a positive class despite fewer examples.

# 3. Preprocessing - Zamira

3.1 Handle missing values

In [ ]:
# according to data exploration in previous section, there are no missing values.

3.2 Feature Engineering (Novel)

In [ ]:
# price per sqft (guard against division by zero)
df['price_per_sqft'] = (df['price'] / df['property_size_sqft']).replace([np.inf, -np.inf], np.nan).fillna(0)

# EMI burden: compute approximate monthly EMI based on loan_amount and loan_tenure_years
# if loan_tenure_years is zero, set EMI monthly to 0
loan_tenure_years_safe = df['loan_tenure_years'].replace({0: np.nan}).fillna(1)
monthly_emi_est = (df['loan_amount'] / (loan_tenure_years_safe * 12)).fillna(0)
df['emi_burden'] = monthly_emi_est / df['customer_salary'].replace({0: np.nan}).fillna(1)

# age of property
current_year = pd.Timestamp.now().year
df['property_age'] = (current_year - df['constructed_year']).clip(lower=0)

# is_new boolean
df['is_new'] = (df['constructed_year'] >= 2015).astype('Int64')

# show new features
display(df[['price_per_sqft','emi_burden','property_age','is_new']].head())

# 4. Exploratory Data Analysis (EDA) - Zamira

In [ ]:
# Visualize distributions of key features
# Plot correlations between features
# Examine relationship between features and target
# Optional: cross-feature analysis, e.g., EMI_to_Salary_ratio vs Decision

4.1 Target distribution plot

In [ ]:
plt.figure(figsize=(5,3))
sns.countplot(x='decision', data=df)
plt.title('Target Distribution (0=Not Buy, 1=Buy)')
plt.show()

4.2 Numeric summary table w/ skew and missing

In [ ]:
num_for_eda = df.select_dtypes(include=['Int64','float']).columns.tolist()
if 'decision' in num_for_eda:
    num_for_eda.remove('decision')

numeric_summary = df[num_for_eda].describe().T
numeric_summary['median'] = df[num_for_eda].median()
numeric_summary['skew'] = df[num_for_eda].skew()
numeric_summary['missing'] = df[num_for_eda].isna().sum()
display(numeric_summary.sort_values('skew', ascending=False).head(20))

4.3 Histograms for key numeric features

In [ ]:
cols_to_plot = ['price','property_size_sqft','price_per_sqft','customer_salary','loan_amount','emi_burden']
cols_to_plot = [c for c in cols_to_plot if c in df.columns]
plt.figure(figsize=(14,8))
for i, col in enumerate(cols_to_plot):
    plt.subplot(2, 3, i+1)
    sns.histplot(df[col], bins=40, kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

4.4 Correlation Matrix (numeric)

In [ ]:
plt.figure(figsize=(12,10))
corr_cols = cols_to_plot + ['property_age','satisfaction_score','neighbourhood_rating','connectivity_score']
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols + ['decision']].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Correlation (selected features + target)")
plt.show()

4.5 Categorical vs target: country and property_type (top features)

In [ ]:
for c in ['country','property_type','furnishing_status']:
    if c in df.columns:
        plt.figure(figsize=(10,4))
        topN = df[c].value_counts().nlargest(12).index
        sns.countplot(x=c, hue='decision', data=df[df[c].isin(topN)])
        plt.xticks(rotation=45)
        plt.title(f"{c} vs decision (top categories)")
        plt.show()

# 5. Handle Class Imbalance - Zamira

In [ ]:
numeric_features = df.select_dtypes(include=['Int64', 'float']).columns.tolist()
if 'decision' in numeric_features:
    numeric_features.remove('decision')

# statistics for emi to income ratio, justifying why we dropped it:
df.groupby('decision')[['emi_to_income_ratio']].agg(['mean','std','min','max'])

In [ ]:
# Check class balance
# Apply resampling methods like SMOTE if necessary
# Confirm new class balance

# drop property_id, not significant
df = df.drop(columns=["property_id"])
df = df.drop(columns=['price_per_sqft', 'emi_burden', 'down_payment', 'emi_to_income_ratio']) # potentially problematic features for dtree, randomforest
# define features for pipeline
all_features = df.columns.tolist()
all_features.remove('decision')
# recompute numerical and categorical lists for pipeline
numeric_features = [c for c in df.select_dtypes(include=['Int64', 'float']).columns.tolist() if c != 'decision']
# include engineered numeric features
extra_nums = ['price_per_sqft', 'emi_burden', 'property_age', 'is_new']
numeric_features = sorted(list(set(numeric_features + [c for c in extra_nums if c in df.columns])))
categorical_features = [c for c in cat_cols if c in df.columns]

print("Numeric features for pipeline:", numeric_features)
print("Categorical features for pipeline:", categorical_features)

# pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

# 6. Train-Test Split - Zamira

In [ ]:
# Split dataset into training and test sets
# Ensure stratification if class imbalance exists

X = df.drop(columns=['decision'])
y = df['decision'].astype(int)  # ensure numpy int

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Preprocessing

In [ ]:
# Fit preprocessor on training data and transform
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# get feature names
feature_names = preprocessor.get_feature_names_out()

print("Processed feature matrix shape:", X_train_proc.shape)
print("Number of processed features:", len(feature_names))

# apply SMOTE on processed training data
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_proc, y_train)
print("After SMOTE, counts:", np.bincount(y_train_res))

# Ensure target is 1D
y_train_res = y_train_res.ravel()
y_train = y_train.ravel()
y_test = y_test.ravel()

# 7. Model Development

In [ ]:
# Helper: function to train and return model
def train_model(clf, X_tr, y_tr):
    clf.fit(X_tr, y_tr)
    return clf

7.1 Baseline Models - Zamira

In [ ]:
# - Logistic Regression
# - Decision Tree
logreg = LogisticRegression(max_iter=1000,
                            class_weight='balanced',
                            random_state=42)
dtree = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42
)

logreg = train_model(logreg, X_train_res, y_train_res)
dtree = train_model(dtree, X_train_res, y_train_res)

7.2 Ensemble and Boosting Models - Zamira

In [ ]:
# rf, gb, and ada

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,            # limit tree depth
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',     # reduce correlation between trees
    class_weight='balanced',
    random_state=42
)
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,       # strong regularization
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,     # stochastic boosting reduces overfitting
    random_state=42
)
ada = AdaBoostClassifier(n_estimators=50,
                         learning_rate=0.1,
                         random_state=42)

rf = train_model(rf, X_train_res, y_train_res)
gb = train_model(gb, X_train_res, y_train_res)
ada = train_model(ada, X_train_res, y_train_res)

7.3 Neural Network - Erik and Zamira

In [ ]:
# Define NN architecture
# Compile and train model
# Evaluate performance on test set
# Compare with ensemble models

# Use GridSearch to find best params
# Do a few params at a time to prevent having large numbers of NNs
#parameters ={
#    'activation':['identity', 'logistic', 'tanh', 'relu'],
#    'solver':['lbfgs', 'sgd', 'adam'],
#    'max_iter':[500]
#}
#mlp = MLPClassifier()
#clf = GridSearchCV(mlp, parameters, cv=5, scoring='accuracy')
#clf.fit(X_train_res, y_train_res)
#print("Best parameters found: ", clf.best_params_)

In [ ]:
# Best hyperparams from above: activation: relu, solver: lbfgs
parameters ={
    'hidden_layer_sizes':[(50,), (50,50)], # smaller hidden layers for lighter ram use
    'activation':['relu'],
    'solver':['adam'], # uses less memory
    'max_iter':[500],
    'alpha':[0.0001, 0.001, 0.01]
}
mlp = MLPClassifier()
clf = GridSearchCV(mlp, parameters, cv=5, scoring='accuracy')
clf.fit(X_train_res, y_train_res)
print("Best parameters found: ", clf.best_params_)

# 8. Model Evaluation and Comparison - Zamira

In [ ]:
# -------------------------------
# 1. Preprocess test (and training subset for SHAP)
# -------------------------------
X_test_processed = preprocessor.transform(X_test)
X_test_dense = X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed

# SHAP subset
X_train_dense_subset = X_train_proc[:2000]

# -------------------------------
# 2. List of models
# -------------------------------
models = {
    'Logistic Regression': logreg,
    'Decision Tree': dtree,
    'Random Forest': rf,
    'Gradient Boosting': gb,
    'AdaBoost': ada,
    'Neural Network': clf.best_estimator_
}

# -------------------------------
# 3. Evaluate metrics
# -------------------------------
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

results = []

for name, model in models.items():
    X_input = X_test_dense if name == 'Neural Network' else X_test_processed

    y_pred = model.predict(X_input)
    y_prob = model.predict_proba(X_input)[:,1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
display(results_df)

# -------------------------------
# 4. Confusion matrices
# -------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18,10))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    X_input = X_test_dense if name == 'Neural Network' else X_test_processed
    y_pred = model.predict(X_input)

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=axes[i], cmap='Blues')
    axes[i].set_title(name)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# -------------------------------
# 5. Feature importance / SHAP
# -------------------------------
import shap
import numpy as np

def get_top_features(model, model_name, top_n=10):
    # Feature names from preprocessor
    features = preprocessor.get_feature_names_out()

    # Logistic Regression
    if model_name == 'Logistic Regression':
        # Make sure coef_ is 1D
        if model.coef_.ndim == 2:
            importances = model.coef_[0]
        else:
            importances = model.coef_

    # Tree-based models
    elif model_name in ['Decision Tree', 'Random Forest', 'Gradient Boosting', 'AdaBoost']:
        importances = model.feature_importances_

    # Neural Network (MLP)
    else:
        background = X_train_proc[:300].toarray() if hasattr(X_train_proc, "toarray") else X_train_proc[:300]
        X_sample = X_train_proc[:300].toarray() if hasattr(X_train_proc, "toarray") else X_train_proc[:300]

        explainer = shap.KernelExplainer(model.predict_proba, background)
        shap_values = explainer.shap_values(X_sample, nsamples=100)
        importances = np.abs(shap_values[1]).mean(axis=0)

    # Safety check
    if len(features) != len(importances):
        print(f"WARNING: Features length {len(features)} != importances length {len(importances)}")
        # Adjust to match
        min_len = min(len(features), len(importances))
        features = features[:min_len]
        importances = importances[:min_len]

    df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    })

    return df.sort_values(by='Importance', ascending=False).head(top_n)


# Top 10 features per model
for name, model in models.items():
    print(f"\nTop features for {name}:")
    display(get_top_features(model, name))
